# 🔧 M1MLOP — Jour 2 : Pipelines ML et Orchestration
**École IT — Master 1 — Unité 3 BigData — Semaine 11**  
**Amadou MAIGA**

---

## 🗂️ Table des matières
1. [Objectifs et contexte](#1)
2. [Pourquoi des pipelines ML ?](#2)
3. [Apache Airflow — Concepts clés](#3)
4. [TP 2.1 — Premier DAG Airflow (séquentiel)](#4)
5. [TP 2.2 — DAG avec parallélisation](#5)
6. [CI/CD pour modèles ML — Tests automatisés](#6)
7. [TP 2.3 — Tests pytest pour modèles ML](#7)
8. [Docker — Conteneuriser un modèle ML](#8)
9. [TP 2.4 — API Flask + Docker](#9)
10. [Kubernetes — Orchestrer les conteneurs](#10)
11. [TP 2.5 — Manifests Kubernetes](#11)
12. [Résumé et bilan du Jour 2](#12)

---

> 📌 **Prérequis** : Jour 1 complété (MLflow, tracking, registry)  
> ⏱️ **Durée estimée** : 7h (matinée + après-midi)

<a id='1'></a>
## 1. 🎯 Objectifs du Jour 2

À la fin de cette journée, tu seras capable de :

- ✅ Concevoir des **pipelines ML automatisés** avec Apache Airflow
- ✅ Orchestrer les tâches : data, preprocessing, training, validation
- ✅ Mettre en place un **CI/CD** pour modèles ML avec tests automatisés
- ✅ **Dockeriser** un modèle ML avec API Flask
- ✅ Déployer avec **Kubernetes** (manifests YAML)

### 🔗 Lien avec le Jour 1

| Jour 1 | Jour 2 |
|---|---|
| Tracking **manuel** des expériences | Tracking **automatisé** dans un pipeline |
| Registry mis à jour **à la main** | Registry mis à jour **par le DAG** |
| Modèle en local | Modèle **conteneurisé** et déployé |

<a id='2'></a>
## 2. ⚠️ Pourquoi des Pipelines ML ?

En Jour 1, tu as tracké les expériences **manuellement**. En production, tout doit être **automatisé**.

> 🏢 **LinkedIn** : LinkedIn réentraîne ses modèles d'appariement emploi-candidat **quotidiennement**. Sans pipeline, les équipes feraient ça 365 fois par an manuellement — impossible. Avec Airflow, c'est automatique et reproductible.

### Un pipeline ML typique en production

```
┌─────────────────────────────────────────────────────────────────┐
│               PIPELINE ML PRODUCTION TYPIQUE                    │
├──────────┬──────────┬──────────┬──────────┬──────────┬──────────┤
│ EXTRACT  │PREPROCESS│ FEATURE  │  TRAIN   │VALIDATE  │  DEPLOY  │
│  DATA    │          │   ENG.   │ (x3 en   │& COMPARE │          │
│   (2h)   │  (30min) │  (1h)    │parallèle)│  (30min) │  (30min) │
└──────────┴──────────┴──────────┴──────────┴──────────┴──────────┘
```

### Problèmes sans orchestration

| Problème | Sans Airflow | Avec Airflow |
|---|---|---|
| L'étape 1 échoue | Tout s'arrête, personne le sait | **Retry automatique** + alerte |
| Lancer 3 modèles en parallèle | Impossible manuellement | **Parallélisation native** |
| Scheduler tous les jours à 2h | Réveil à 2h du matin 😴 | **CRON intégré** |
| Rollback si validation échoue | Manuel, lent | **Automatique** |

<a id='3'></a>
## 3. 🌬️ Apache Airflow — Concepts Clés

**Apache Airflow** est un scheduler de workflows open-source créé par **Airbnb** en 2014.

> 🚀 **En production** : Spotify, Netflix, Airbnb, Twitter, Stripe utilisent Airflow pour orchestrer des milliers de pipelines quotidiens.

### Les 5 concepts fondamentaux

```
┌─────────────────────────────────────────────────────────────┐
│                  AIRFLOW — CONCEPTS CLÉS                    │
├────────────┬────────────────────────────────────────────────┤
│ DAG        │ Directed Acyclic Graph : le pipeline complet   │
│ Task       │ Une unité d'exécution (1 étape du pipeline)    │
│ Operator   │ Type de task (Python, Bash, SQL, Spark...)     │
│ Scheduler  │ Lance les DAGs selon un planning CRON          │
│ Executor   │ Exécute les tasks (local, Celery, K8s...)      │
└────────────┴────────────────────────────────────────────────┘
```

### DAG — Directed Acyclic Graph

```
                    ┌─────────────┐
                    │   EXTRACT   │
                    └──────┬──────┘
                           │
                    ┌──────▼──────┐
                    │  PREPROCESS │
                    └──────┬──────┘
              ┌────────────┼────────────┐
       ┌──────▼──────┐     │     ┌──────▼──────┐
       │  TRAIN RF   │     │     │  TRAIN SVM  │
       └──────┬──────┘     │     └──────┬──────┘
              └────────────┼────────────┘
                    ┌──────▼──────┐
                    │   COMPARE   │
                    └──────┬──────┘
                    ┌──────▼──────┐
                    │   DEPLOY    │
                    └─────────────┘
```

### Installation Airflow

```bash
pip install apache-airflow
export AIRFLOW_HOME=~/airflow
airflow db init

# Lancer l'UI (port 8080)
airflow webserver --port 8080

# Dans un autre terminal, lancer le scheduler
airflow scheduler
```

In [ ]:
# ============================================================
# 📦 INSTALLATION DES DÉPENDANCES JOUR 2
# ============================================================
import sys

!{sys.executable} -m pip install scikit-learn pandas numpy matplotlib pytest --quiet

print("✅ Dépendances installées !")
print("")
print("ℹ️  Pour Airflow (nécessite un terminal dédié) :")
print("   pip install apache-airflow")
print("   export AIRFLOW_HOME=~/airflow")
print("   airflow db init")

In [ ]:
# ============================================================
# 📚 IMPORTS GLOBAUX
# ============================================================
import numpy as np
import pandas as pd
import os
import pickle
import json
import time
import random
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("✅ Imports OK")
print(f"   NumPy  : {np.__version__}")
print(f"   Pandas : {pd.__version__}")

<a id='4'></a>
## 4. 🧪 TP 2.1 — Premier DAG Airflow (Pipeline Séquentiel)

### Objectif
Créer un DAG Airflow complet avec **4 tasks en séquence** :
`extract → preprocess → train → validate`

### Structure d'un DAG Python

```python
# Anatomie d'un DAG Airflow

from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta

# 1. Définir les fonctions (le travail réel)
def ma_fonction():
    print("Je fais quelque chose")

# 2. Configurer le DAG
default_args = {'owner': 'team', 'retries': 2}
dag = DAG('mon_dag', default_args=default_args, schedule_interval='@daily')

# 3. Créer les tasks
task = PythonOperator(task_id='ma_task', python_callable=ma_fonction, dag=dag)

# 4. Définir les dépendances
task_a >> task_b >> task_c  # séquentiel
task_a >> [task_b, task_c] >> task_d  # parallèle
```

> 🏭 **Stripe (Paiements)** : Stripe utilise Airflow pour entraîner ses modèles de détection de fraude **4 fois par jour**. Airflow orchestre ce timing automatiquement.

In [ ]:
# ============================================================
# TP 2.1 — SIMULATION D'UN PIPELINE SÉQUENTIEL
# (Simule ce que ferait Airflow, sans avoir besoin d'installer Airflow)
# ============================================================

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ---- TASK 1 : extract_data ----
def extract_data():
    """Simule l'extraction depuis un data warehouse."""
    logger.info("[TASK 1] 📥 Extracting data from warehouse...")
    
    # En prod : connexion DB, API, S3...
    iris = load_iris()
    df = pd.DataFrame(iris.data, columns=iris.feature_names)
    df['target'] = iris.target
    
    # Sauvegarder pour la prochaine task
    df.to_csv('/tmp/extracted.csv', index=False)
    
    logger.info(f"[TASK 1] ✅ Extracted {len(df)} rows → /tmp/extracted.csv")
    return len(df)

# ---- TASK 2 : preprocess_data ----
def preprocess_data():
    """Nettoyage et préprocessing des données."""
    logger.info("[TASK 2] 🔧 Preprocessing data...")
    
    df = pd.read_csv('/tmp/extracted.csv')
    
    # Nettoyage
    before = len(df)
    df = df.dropna()  # Supprimer les NaN
    
    # Normalisation min-max
    feature_cols = [c for c in df.columns if c != 'target']
    df[feature_cols] = (df[feature_cols] - df[feature_cols].min()) / \
                       (df[feature_cols].max() - df[feature_cols].min())
    
    df.to_csv('/tmp/preprocessed.csv', index=False)
    
    logger.info(f"[TASK 2] ✅ Preprocessed: {before} → {len(df)} rows (normalized)")
    return len(df)

# ---- TASK 3 : train_model ----
def train_model():
    """Entraîne le modèle et sauvegarde."""
    logger.info("[TASK 3] 🤖 Training model...")
    
    df = pd.read_csv('/tmp/preprocessed.csv')
    X = df.drop('target', axis=1).values
    y = df['target'].values
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    
    model = RandomForestClassifier(n_estimators=100, random_state=SEED)
    model.fit(X_train, y_train)
    
    # Sauvegarder le modèle et les données de test
    with open('/tmp/model.pkl', 'wb') as f:
        pickle.dump(model, f)
    
    np.save('/tmp/X_test.npy', X_test)
    np.save('/tmp/y_test.npy', y_test)
    
    logger.info(f"[TASK 3] ✅ Model trained on {len(X_train)} samples → /tmp/model.pkl")
    return len(X_train)

# ---- TASK 4 : validate_model ----
def validate_model(accuracy_threshold=0.85):
    """Valide les performances du modèle."""
    logger.info("[TASK 4] 📊 Validating model...")
    
    with open('/tmp/model.pkl', 'rb') as f:
        model = pickle.load(f)
    
    X_test = np.load('/tmp/X_test.npy')
    y_test = np.load('/tmp/y_test.npy')
    
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    # Critère de validation
    if accuracy >= accuracy_threshold:
        status = "✅ PASSED"
        decision = "DEPLOY"
    else:
        status = "❌ FAILED"
        decision = "ROLLBACK"
        raise ValueError(f"Accuracy {accuracy:.4f} < threshold {accuracy_threshold}")
    
    logger.info(f"[TASK 4] {status} | Accuracy: {accuracy:.4f} | F1: {f1:.4f} | Decision: {decision}")
    return {'accuracy': accuracy, 'f1': f1, 'decision': decision}

print("✅ Fonctions du pipeline définies !")

In [ ]:
# ============================================================
# EXÉCUTION DU PIPELINE SÉQUENTIEL
# (Simule ce que Airflow ferait automatiquement)
# ============================================================

print("=" * 65)
print("       🚀 EXÉCUTION DU PIPELINE ML SÉQUENTIEL")
print("=" * 65)
print()

pipeline_results = {}
pipeline_start = time.time()

tasks = [
    ("extract_data",    extract_data),
    ("preprocess_data", preprocess_data),
    ("train_model",     train_model),
    ("validate_model",  validate_model),
]

for task_name, task_fn in tasks:
    task_start = time.time()
    try:
        result = task_fn()
        duration = time.time() - task_start
        pipeline_results[task_name] = {"status": "SUCCESS", "duration": duration, "result": result}
        print(f"  ✅ {task_name:20s} | {duration:.2f}s | SUCCESS")
    except Exception as e:
        duration = time.time() - task_start
        pipeline_results[task_name] = {"status": "FAILED", "duration": duration, "error": str(e)}
        print(f"  ❌ {task_name:20s} | {duration:.2f}s | FAILED: {e}")
        print("  ⚠️  Pipeline stoppé (comportement Airflow)")
        break

total_time = time.time() - pipeline_start
print()
print(f"{'='*65}")
print(f"  ⏱️  Durée totale : {total_time:.2f}s")
success_count = sum(1 for v in pipeline_results.values() if v['status'] == 'SUCCESS')
print(f"  📊 Tasks réussies : {success_count}/{len(tasks)}")
print(f"{'='*65}")

In [ ]:
# ============================================================
# GÉNÉRER LE FICHIER DAG AIRFLOW RÉEL
# (à placer dans ~/airflow/dags/ pour Airflow)
# ============================================================

dag_code = '''
# ============================================================
# DAG Airflow — TP 2.1 — Pipeline ML Séquentiel
# Amadou MAIGA — M1MLOP — École IT
# 
# Pour utiliser ce DAG :
#   1. Copier ce fichier dans ~/airflow/dags/
#   2. Lancer airflow webserver --port 8080
#   3. Lancer airflow scheduler
#   4. Ouvrir http://localhost:8080
# ============================================================

from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator
import pandas as pd
import numpy as np
import pickle
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

SEED = 42

def extract_data():
    iris = load_iris()
    df = pd.DataFrame(iris.data, columns=iris.feature_names)
    df[\'target\'] = iris.target
    df.to_csv(\'/tmp/extracted.csv\', index=False)
    print(f"Extracted {len(df)} rows")

def preprocess_data():
    df = pd.read_csv(\'/tmp/extracted.csv\')
    df = df.dropna()
    feature_cols = [c for c in df.columns if c != \'target\']
    df[feature_cols] = (df[feature_cols] - df[feature_cols].min()) / \\\
                       (df[feature_cols].max() - df[feature_cols].min())
    df.to_csv(\'/tmp/preprocessed.csv\', index=False)
    print(f"Preprocessed {len(df)} rows")

def train_model():
    df = pd.read_csv(\'/tmp/preprocessed.csv\')
    X = df.drop(\'target\', axis=1).values
    y = df[\'target\'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
    model = RandomForestClassifier(n_estimators=100, random_state=SEED)
    model.fit(X_train, y_train)
    with open(\'/tmp/model.pkl\', \'wb\') as f:
        pickle.dump(model, f)
    np.save(\'/tmp/X_test.npy\', X_test)
    np.save(\'/tmp/y_test.npy\', y_test)
    print(f"Model trained on {len(X_train)} samples")

def validate_model():
    with open(\'/tmp/model.pkl\', \'rb\') as f:
        model = pickle.load(f)
    X_test = np.load(\'/tmp/X_test.npy\')
    y_test = np.load(\'/tmp/y_test.npy\')
    accuracy = accuracy_score(y_test, model.predict(X_test))
    print(f"Model accuracy: {accuracy:.4f}")
    if accuracy < 0.85:
        raise ValueError(f"Accuracy {accuracy:.4f} too low!")

# Définir le DAG
default_args = {
    \'owner\': \'amadou-maiga\',
    \'retries\': 2,
    \'retry_delay\': timedelta(minutes=5),
    \'start_date\': datetime(2025, 12, 7),
    \'email_on_failure\': False,
}

dag = DAG(
    \'ml_training_pipeline_tp21\',
    default_args=default_args,
    description=\'TP 2.1 - Pipeline ML sequentiel : extract → preprocess → train → validate\',
    schedule_interval=\'0 2 * * *\',  # Tous les jours à 2h du matin
    catchup=False,
    tags=[\'mlops\', \'training\', \'m1mlop\']
)

# Définir les tasks
t1 = PythonOperator(task_id=\'extract_data\',     python_callable=extract_data,    dag=dag)
t2 = PythonOperator(task_id=\'preprocess_data\',  python_callable=preprocess_data, dag=dag)
t3 = PythonOperator(task_id=\'train_model\',      python_callable=train_model,     dag=dag)
t4 = PythonOperator(task_id=\'validate_model\',   python_callable=validate_model,  dag=dag)

# Dépendances séquentielles
t1 >> t2 >> t3 >> t4
'''

os.makedirs('airflow_dags', exist_ok=True)
with open('airflow_dags/dag_tp21_sequential.py', 'w') as f:
    f.write(dag_code)

print("✅ Fichier DAG généré : airflow_dags/dag_tp21_sequential.py")
print()
print("📋 Pour l'utiliser avec Airflow :")
print("   cp airflow_dags/dag_tp21_sequential.py ~/airflow/dags/")
print("   airflow webserver --port 8080")
print("   airflow scheduler")
print("   → http://localhost:8080")

<a id='5'></a>
## 5. 🧪 TP 2.2 — DAG avec Parallélisation

### Objectif
Entraîner **3 modèles en parallèle**, puis comparer et déployer le meilleur.

### Syntaxe Airflow pour la parallélisation

```python
# Séquentiel
t1 >> t2 >> t3 >> t4

# Parallèle : t2, t3, t4 tournent en même temps
t1 >> [t2, t3, t4] >> t5
```

> 💡 **Amazon** : Amazon parallélise l'entraînement de **1000+ modèles** de recommandation simultanément avec Airflow. Sans parallélisation, ce serait 1000x plus lent.

In [ ]:
# ============================================================
# TP 2.2 — PIPELINE AVEC PARALLÉLISATION
# ============================================================
import concurrent.futures

def train_random_forest():
    """Task parallèle 1 : Random Forest."""
    df = pd.read_csv('/tmp/preprocessed.csv')
    X = df.drop('target', axis=1).values
    y = df['target'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
    
    model = RandomForestClassifier(n_estimators=100, random_state=SEED)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    f1  = f1_score(y_test, model.predict(X_test), average='weighted')
    
    with open('/tmp/model_rf.pkl', 'wb') as f:
        pickle.dump(model, f)
    
    result = {'model': 'RandomForest', 'accuracy': acc, 'f1': f1, 'path': '/tmp/model_rf.pkl'}
    logger.info(f"[RF]  Accuracy={acc:.4f} | F1={f1:.4f}")
    return result

def train_gradient_boosting():
    """Task parallèle 2 : Gradient Boosting."""
    df = pd.read_csv('/tmp/preprocessed.csv')
    X = df.drop('target', axis=1).values
    y = df['target'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
    
    model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=SEED)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    f1  = f1_score(y_test, model.predict(X_test), average='weighted')
    
    with open('/tmp/model_gb.pkl', 'wb') as f:
        pickle.dump(model, f)
    
    result = {'model': 'GradientBoosting', 'accuracy': acc, 'f1': f1, 'path': '/tmp/model_gb.pkl'}
    logger.info(f"[GB]  Accuracy={acc:.4f} | F1={f1:.4f}")
    return result

def train_svm():
    """Task parallèle 3 : SVM."""
    df = pd.read_csv('/tmp/preprocessed.csv')
    X = df.drop('target', axis=1).values
    y = df['target'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
    
    model = SVC(C=1.0, kernel='rbf', random_state=SEED, probability=True)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    f1  = f1_score(y_test, model.predict(X_test), average='weighted')
    
    with open('/tmp/model_svm.pkl', 'wb') as f:
        pickle.dump(model, f)
    
    result = {'model': 'SVM', 'accuracy': acc, 'f1': f1, 'path': '/tmp/model_svm.pkl'}
    logger.info(f"[SVM] Accuracy={acc:.4f} | F1={f1:.4f}")
    return result

def compare_and_select(results):
    """Task finale : comparer les 3 modèles et sélectionner le meilleur."""
    best = max(results, key=lambda r: r['f1'])
    logger.info(f"[COMPARE] 🏆 Best model: {best['model']} | F1={best['f1']:.4f}")
    
    # Copier le meilleur modèle
    import shutil
    shutil.copy(best['path'], '/tmp/best_model.pkl')
    logger.info(f"[COMPARE] ✅ Best model saved → /tmp/best_model.pkl")
    return best

print("✅ Fonctions de parallélisation définies !")

In [ ]:
# ============================================================
# EXÉCUTION DU PIPELINE PARALLÈLE
# ============================================================

print("=" * 65)
print("       🚀 PIPELINE PARALLÈLE — 3 MODÈLES EN MÊME TEMPS")
print("=" * 65)
print()

# Step 1 : Extract + Preprocess (séquentiel)
print("📥 ÉTAPE 1 : Extract & Preprocess (séquentiel)")
extract_data()
preprocess_data()
print()

# Step 2 : Entraîner 3 modèles EN PARALLÈLE
print("🔀 ÉTAPE 2 : Entraînement parallèle (3 modèles simultanés)")
parallel_start = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    futures = {
        executor.submit(train_random_forest): 'RandomForest',
        executor.submit(train_gradient_boosting): 'GradientBoosting',
        executor.submit(train_svm): 'SVM',
    }
    
    parallel_results = []
    for future in concurrent.futures.as_completed(futures):
        model_name = futures[future]
        result = future.result()
        parallel_results.append(result)
        print(f"  ✅ {result['model']:20s} | Accuracy: {result['accuracy']:.4f} | F1: {result['f1']:.4f}")

parallel_time = time.time() - parallel_start
print(f"  ⏱️  3 modèles entraînés en {parallel_time:.2f}s (en parallèle)")
print()

# Step 3 : Comparer et sélectionner
print("🏆 ÉTAPE 3 : Comparaison et sélection")
best_model = compare_and_select(parallel_results)
print()

# Résultats
print("=" * 65)
results_df = pd.DataFrame(parallel_results).sort_values('f1', ascending=False)
print(results_df[['model', 'accuracy', 'f1']].to_string(index=False))
print("=" * 65)
print(f"  🥇 Meilleur modèle : {best_model['model']} (F1 = {best_model['f1']:.4f})")
print("=" * 65)

In [ ]:
# ============================================================
# VISUALISATION DU DAG PARALLÈLE
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- Graphique 1 : Comparaison des modèles ----
models  = [r['model'] for r in parallel_results]
accs    = [r['accuracy'] for r in parallel_results]
f1s     = [r['f1'] for r in parallel_results]
x       = np.arange(len(models))
width   = 0.35

bars1 = axes[0].bar(x - width/2, accs, width, label='Accuracy', color='steelblue', alpha=0.8)
bars2 = axes[0].bar(x + width/2, f1s,  width, label='F1-Score',  color='coral',    alpha=0.8)

axes[0].set_xlabel('Modèle')
axes[0].set_ylabel('Score')
axes[0].set_title('📊 Comparaison des 3 modèles parallèles')
axes[0].set_xticks(x)
axes[0].set_xticklabels(['RF', 'GB', 'SVM'])
axes[0].set_ylim(0.85, 1.02)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

# ---- Graphique 2 : Schéma du DAG ----
axes[1].set_xlim(0, 10)
axes[1].set_ylim(0, 10)
axes[1].set_title('🔀 Architecture du DAG Airflow (TP 2.2)')
axes[1].axis('off')

def draw_box(ax, x, y, w, h, text, color):
    rect = mpatches.FancyBboxPatch((x-w/2, y-h/2), w, h,
                                    boxstyle="round,pad=0.1",
                                    facecolor=color, edgecolor='gray', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=9, fontweight='bold')

draw_box(axes[1], 5, 9,   3, 0.7, '📥 EXTRACT',        '#AED6F1')
draw_box(axes[1], 5, 7.5, 3, 0.7, '🔧 PREPROCESS',     '#AED6F1')
draw_box(axes[1], 2, 5.5, 2.5, 0.7, '🌲 TRAIN RF',     '#A9DFBF')
draw_box(axes[1], 5, 5.5, 2.5, 0.7, '📈 TRAIN GB',     '#A9DFBF')
draw_box(axes[1], 8, 5.5, 2.5, 0.7, '🔷 TRAIN SVM',    '#A9DFBF')
draw_box(axes[1], 5, 3.5, 3,   0.7, '🏆 COMPARE',       '#F9E79F')
draw_box(axes[1], 5, 2,   3,   0.7, '🚀 DEPLOY BEST',   '#F1948A')

# Flèches
for (x1,y1,x2,y2) in [(5,8.65,5,7.85),(5,7.15,2,5.85),(5,7.15,5,5.85),(5,7.15,8,5.85),
                        (2,5.15,5,3.85),(5,5.15,5,3.85),(8,5.15,5,3.85),(5,3.15,5,2.35)]:
    axes[1].annotate('', xy=(x2,y2), xytext=(x1,y1),
                     arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

axes[1].text(6.5, 6.3, '// parallèle //', color='green', fontsize=8, style='italic')

plt.tight_layout()
plt.savefig('dag_parallel_visualization.png', dpi=100, bbox_inches='tight')
plt.show()
print("📸 Graphique sauvegardé : dag_parallel_visualization.png")

<a id='6'></a>
## 6. 🔄 CI/CD pour Modèles ML

**CI/CD = Continuous Integration / Continuous Deployment**

| Concept | Définition | Pour ML |
|---|---|---|
| **CI** | Tester automatiquement chaque changement | Tests du modèle + pipeline |
| **CD** | Déployer automatiquement si les tests passent | Déploiement du modèle en prod |

> 🚀 **Google (YouTube)** : Google déploie les modèles de ranking YouTube **plusieurs fois par jour** avec CI/CD. Chaque modèle passe par **100+ tests automatisés** avant la production.

### Les 3 types de tests ML

```
┌─────────────────────────────────────────────────────┐
│              PYRAMIDE DE TESTS ML                   │
│                                                     │
│              ┌─────────────┐                        │
│              │ INTEGRATION │  Pipeline complet      │
│           ┌──┴─────────────┴──┐                     │
│           │   MODEL TESTS     │  Performance, drift │
│        ┌──┴───────────────────┴──┐                  │
│        │      UNIT TESTS         │  Fonctions seules │
│        └─────────────────────────┘                  │
└─────────────────────────────────────────────────────┘
```

<a id='7'></a>
## 7. 🧪 TP 2.3 — Tests pytest pour Modèles ML

### Objectif
Écrire des tests automatisés (unit + model + integration) avec **pytest**.

> 📊 **Uber (Surge Pricing)** : Les modèles Uber sont testés automatiquement contre **10 scénarios réels** avant déploiement. Un test échoue = pas de déploiement ce jour-là.

In [ ]:
# ============================================================
# TP 2.3 — TESTS AUTOMATISÉS AVEC PYTEST
# ============================================================

# ---- Fonctions utilitaires à tester ----
def preprocess(df):
    """Preprocessing : supprimer les NaN."""
    return df.dropna()

def validate_accuracy(model, X_test, y_test, threshold=0.80):
    """Valide l'accuracy d'un modèle."""
    acc = model.score(X_test, y_test)
    return acc >= threshold, acc

def predict_single(model, features):
    """Prédiction sur une seule observation."""
    return model.predict([features])[0]

# ============================================================
# UNIT TESTS
# ============================================================
print("🔬 UNIT TESTS")
print("-" * 50)

def test_preprocessing_removes_nan():
    """Test 1 : Le preprocessing supprime bien les NaN."""
    df = pd.DataFrame({'a': [1, 2, None], 'b': [4, None, 6]})
    result = preprocess(df)
    assert len(result) == 1, f"Expected 1 row, got {len(result)}"
    assert result.isna().sum().sum() == 0, "NaN still present after preprocessing"
    print("  ✅ test_preprocessing_removes_nan")

def test_preprocessing_preserves_clean_data():
    """Test 2 : Le preprocessing ne supprime pas les données propres."""
    df = pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]})
    result = preprocess(df)
    assert len(result) == 3, f"Expected 3 rows, got {len(result)}"
    print("  ✅ test_preprocessing_preserves_clean_data")

def test_preprocessing_empty_df():
    """Test 3 : Le preprocessing gère un DataFrame vide."""
    df = pd.DataFrame({'a': [], 'b': []})
    result = preprocess(df)
    assert len(result) == 0
    print("  ✅ test_preprocessing_empty_df")

test_preprocessing_removes_nan()
test_preprocessing_preserves_clean_data()
test_preprocessing_empty_df()
print()

In [ ]:
# ============================================================
# MODEL TESTS
# ============================================================
print("🤖 MODEL TESTS")
print("-" * 50)

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
test_model = RandomForestClassifier(n_estimators=50, random_state=SEED)
test_model.fit(X_train, y_train)

def test_model_accuracy_above_threshold():
    """Test 4 : Le modèle atteint au moins 80% d'accuracy."""
    passed, acc = validate_accuracy(test_model, X_test, y_test, threshold=0.80)
    assert passed, f"Accuracy {acc:.4f} < 0.80"
    print(f"  ✅ test_model_accuracy_above_threshold (accuracy={acc:.4f})")

def test_model_output_shape():
    """Test 5 : Le modèle retourne le bon nombre de prédictions."""
    preds = test_model.predict(X_test)
    assert len(preds) == len(X_test), f"Expected {len(X_test)}, got {len(preds)}"
    print(f"  ✅ test_model_output_shape ({len(preds)} predictions)")

def test_model_valid_classes():
    """Test 6 : Les prédictions sont dans les classes valides [0, 1, 2]."""
    preds = test_model.predict(X_test)
    valid_classes = set([0, 1, 2])
    assert set(preds).issubset(valid_classes), f"Invalid classes: {set(preds) - valid_classes}"
    print(f"  ✅ test_model_valid_classes (classes={set(preds)})")

def test_model_single_prediction():
    """Test 7 : Prédiction sur une seule observation."""
    sample = X_test[0]
    pred = predict_single(test_model, sample)
    assert pred in [0, 1, 2], f"Invalid prediction: {pred}"
    print(f"  ✅ test_model_single_prediction (pred={pred})")

def test_model_robustness_extreme_values():
    """Test 8 : Le modèle gère des valeurs extrêmes sans crash."""
    extreme_input = np.array([[0.0, 0.0, 0.0, 0.0]])
    try:
        pred = test_model.predict(extreme_input)
        assert len(pred) == 1
        print(f"  ✅ test_model_robustness_extreme_values (pred={pred[0]})")
    except Exception as e:
        print(f"  ❌ test_model_robustness_extreme_values FAILED: {e}")
        raise

test_model_accuracy_above_threshold()
test_model_output_shape()
test_model_valid_classes()
test_model_single_prediction()
test_model_robustness_extreme_values()
print()

In [ ]:
# ============================================================
# INTEGRATION TEST + GÉNÉRATION DU FICHIER pytest
# ============================================================
print("🔗 INTEGRATION TEST")
print("-" * 50)

def test_full_pipeline_integration():
    """Test 9 : Le pipeline complet s'exécute sans erreur."""
    # 1. Extract
    iris = load_iris()
    df = pd.DataFrame(iris.data, columns=iris.feature_names)
    df['target'] = iris.target
    assert len(df) > 0, "No data extracted"
    
    # 2. Preprocess
    df_clean = preprocess(df)
    assert len(df_clean) > 0, "No data after preprocessing"
    
    # 3. Train
    X = df_clean.drop('target', axis=1).values
    y = df_clean['target'].values
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED)
    m = RandomForestClassifier(n_estimators=10, random_state=SEED)
    m.fit(X_tr, y_tr)
    
    # 4. Validate
    passed, acc = validate_accuracy(m, X_te, y_te, threshold=0.70)
    assert passed, f"Pipeline integration test failed: accuracy {acc:.4f} < 0.70"
    
    print(f"  ✅ test_full_pipeline_integration (accuracy={acc:.4f})")

test_full_pipeline_integration()
print()
print("=" * 50)
print("  📊 BILAN : 9/9 tests PASSED ✅")
print("=" * 50)

# Générer le fichier pytest
pytest_code = '''# test_pipeline.py — TP 2.3 — M1MLOP — Amadou MAIGA
# Lancer avec : pytest test_pipeline.py -v

import pytest
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

SEED = 42

@pytest.fixture
def trained_model():
    X, y = load_iris(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
    model = RandomForestClassifier(n_estimators=50, random_state=SEED)
    model.fit(X_train, y_train)
    return model, X_test, y_test

def test_preprocessing_removes_nan():
    df = pd.DataFrame({"a": [1, 2, None], "b": [4, None, 6]})
    result = df.dropna()
    assert len(result) == 1
    assert result.isna().sum().sum() == 0

def test_model_accuracy(trained_model):
    model, X_test, y_test = trained_model
    acc = model.score(X_test, y_test)
    assert acc >= 0.80, f"Accuracy {acc:.4f} < 0.80"

def test_model_output_shape(trained_model):
    model, X_test, y_test = trained_model
    preds = model.predict(X_test)
    assert len(preds) == len(X_test)

def test_model_valid_classes(trained_model):
    model, X_test, _ = trained_model
    preds = model.predict(X_test)
    assert set(preds).issubset({0, 1, 2})

def test_model_single_prediction(trained_model):
    model, X_test, _ = trained_model
    pred = model.predict(X_test[:1])
    assert len(pred) == 1
    assert pred[0] in [0, 1, 2]
'''

with open('test_pipeline.py', 'w') as f:
    f.write(pytest_code)

print()
print("✅ Fichier pytest généré : test_pipeline.py")
print("   Pour lancer les tests : pytest test_pipeline.py -v")

<a id='8'></a>
## 8. 🐳 Docker — Conteneuriser un Modèle ML

Docker crée une **"boîte"** (conteneur) avec ton code + dépendances + OS minimal.

**Avantage** : *"Ça marche sur mon ordi"* → *"Ça marche partout"* (dev, test, prod, cloud).

### Architecture d'une API ML dockerisée

```
┌─────────────────────────────────────────────────┐
│              CONTENEUR DOCKER                   │
│  ┌─────────────────────────────────────────┐    │
│  │  app.py (API Flask)                     │    │
│  │    POST /predict → model.predict()      │    │
│  │    GET  /health  → {status: ok}         │    │
│  ├─────────────────────────────────────────┤    │
│  │  model.pkl (RandomForest entraîné)      │    │
│  ├─────────────────────────────────────────┤    │
│  │  requirements.txt                        │    │
│  │    flask, scikit-learn, pandas...       │    │
│  └─────────────────────────────────────────┘    │
│  PORT 5000 exposé                               │
└─────────────────────────────────────────────────┘
         ↑
  docker run -p 5000:5000 ml-api:1.0
```

> 🔧 **AWS, Google Cloud, Azure** : Tous les cloud providers utilisent Docker. Une fois ton modèle en Docker, tu peux le déployer sur n'importe quel cloud.

<a id='9'></a>
## 9. 🧪 TP 2.4 — API Flask + Docker

In [ ]:
# ============================================================
# ÉTAPE 1 : Entraîner et sauvegarder le modèle
# ============================================================

print("🤖 Entraînement du modèle pour l'API...")

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

api_model = RandomForestClassifier(n_estimators=100, random_state=SEED)
api_model.fit(X_train, y_train)

accuracy = api_model.score(X_test, y_test)
print(f"  Accuracy : {accuracy:.4f}")

# Sauvegarder avec les métadonnées
model_data = {
    'model': api_model,
    'feature_names': load_iris().feature_names,
    'target_names': load_iris().target_names.tolist(),
    'accuracy': accuracy,
    'version': '1.0'
}

os.makedirs('docker_app', exist_ok=True)
with open('docker_app/model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print(f"✅ Modèle sauvegardé : docker_app/model.pkl")

In [ ]:
# ============================================================
# ÉTAPE 2 : Générer l'API Flask
# ============================================================

flask_app = '''# app.py — API Flask pour modèle ML
# M1MLOP — Amadou MAIGA — TP 2.4
# Lancer : python app.py
# Tester : curl -X POST http://localhost:5000/predict \\
#           -H "Content-Type: application/json" \\
#           -d \'{"features": [5.1, 3.5, 1.4, 0.2]}\'\n
from flask import Flask, request, jsonify
import pickle
import numpy as np
import logging
from datetime import datetime

app = Flask(__name__)
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Charger le modèle UNE SEULE FOIS au démarrage
with open(\'model.pkl\', \'rb\') as f:
    model_data = pickle.load(f)

model        = model_data[\'model\']
target_names = model_data[\'target_names\']
model_version = model_data[\'version\']

logger.info(f"Model loaded. Version: {model_version} | Accuracy: {model_data[\'accuracy\']:.4f}")

@app.route(\'/health\', methods=[\'GET\'])
def health():
    """Health check — utilisé par Kubernetes."""
    return jsonify({
        \'status\': \'healthy\',
        \'model_version\': model_version,
        \'timestamp\': datetime.utcnow().isoformat()
    })

@app.route(\'/predict\', methods=[\'POST\'])
def predict():
    """
    Endpoint de prédiction.
    Input  : {"features": [5.1, 3.5, 1.4, 0.2]}
    Output : {"prediction": 0, "species": "setosa", "confidence": 0.97}
    """
    try:
        data = request.get_json()
        
        if not data or \'features\' not in data:
            return jsonify({\'error\': \'Missing field: features\'}), 400
        
        features = np.array(data[\'features\']).reshape(1, -1)
        
        if features.shape[1] != 4:
            return jsonify({\'error\': f\'Expected 4 features, got {features.shape[1]}\'}), 400
        
        prediction  = int(model.predict(features)[0])
        probas      = model.predict_proba(features)[0]
        confidence  = float(np.max(probas))
        species     = target_names[prediction]
        
        logger.info(f"Prediction: {species} (confidence={confidence:.3f})")
        
        return jsonify({
            \'prediction\':   prediction,
            \'species\'  :   species,
            \'confidence\'  : round(confidence, 3),
            \'probabilities\': {name: round(float(p), 3) for name, p in zip(target_names, probas)},
            \'model_version\': model_version
        })
    
    except Exception as e:
        logger.error(f"Prediction error: {str(e)}")
        return jsonify({\'error\': str(e)}), 500

@app.route(\'/info\', methods=[\'GET\'])
def info():
    """Informations sur le modèle déployé."""
    return jsonify({
        \'model_type\'   : type(model).__name__,
        \'model_version\': model_version,
        \'accuracy\'     : round(model_data[\'accuracy\'], 4),
        \'classes\'      : target_names,
        \'features\'     : model_data[\'feature_names\']
    })

if __name__ == \'__main__\':
    app.run(host=\'0.0.0.0\', port=5000, debug=False)
'''

with open('docker_app/app.py', 'w') as f:
    f.write(flask_app)

print("✅ API Flask générée : docker_app/app.py")

In [ ]:
# ============================================================
# ÉTAPE 3 : Générer le Dockerfile et requirements
# ============================================================

dockerfile = '''# Dockerfile — API ML Iris
# M1MLOP — Amadou MAIGA — TP 2.4

# Image de base légère Python
FROM python:3.10-slim

# Répertoire de travail dans le conteneur
WORKDIR /app

# Copier requirements en premier (optimise le cache Docker)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copier le code et le modèle
COPY app.py .
COPY model.pkl .

# Exposer le port
EXPOSE 5000

# Health check Kubernetes
HEALTHCHECK --interval=30s --timeout=10s --start-period=5s \\\
  CMD curl -f http://localhost:5000/health || exit 1

# Commande de démarrage
CMD ["python", "app.py"]
'''

requirements_docker = '''flask==3.0.0
scikit-learn==1.3.2
numpy==1.24.0
pandas==2.0.0
'''

with open('docker_app/Dockerfile', 'w') as f:
    f.write(dockerfile)

with open('docker_app/requirements.txt', 'w') as f:
    f.write(requirements_docker)

print("✅ Dockerfile généré : docker_app/Dockerfile")
print("✅ requirements.txt  : docker_app/requirements.txt")
print()
print("📋 Commandes Docker :")
print("   cd docker_app")
print("   docker build -t ml-api:1.0 .")
print("   docker run -p 5000:5000 ml-api:1.0")
print()
print("📋 Tester l'API :")
print('   curl -X POST http://localhost:5000/predict \\')
print('     -H "Content-Type: application/json" \\')
print('     -d \'{"features": [5.1, 3.5, 1.4, 0.2]}\'')
print()

# Lister les fichiers générés
print("📁 Fichiers dans docker_app/ :")
for f in os.listdir('docker_app'):
    size = os.path.getsize(f'docker_app/{f}')
    print(f"   {f:20s} {size:>8,} bytes")

<a id='10'></a>
## 10. ☸️ Kubernetes — Orchestrer les Conteneurs

**Kubernetes (K8s)** gère les conteneurs Docker en production.

### Problèmes résolus par Kubernetes

| Problème | Solution K8s |
|---|---|
| Besoin de 10 copies du modèle | `replicas: 10` dans le Deployment |
| Un conteneur crash | K8s en relance un autre **automatiquement** |
| Trop de requêtes | **HPA** lance plus de pods automatiquement |
| Nouvelle version à déployer | **Rolling Update** sans downtime |

### Concepts K8s

```
┌─────────────────────────────────────────────────────┐
│                  CLUSTER KUBERNETES                  │
│  ┌─────────────────────────────────────────────┐    │
│  │  Deployment (ml-api)                        │    │
│  │  ┌──────────┐ ┌──────────┐ ┌──────────┐    │    │
│  │  │  Pod 1   │ │  Pod 2   │ │  Pod 3   │    │    │
│  │  │ml-api:1.0│ │ml-api:1.0│ │ml-api:1.0│    │    │
│  │  └──────────┘ └──────────┘ └──────────┘    │    │
│  └─────────────────────────────────────────────┘    │
│         ↑ Load Balancer (Service)                    │
│  ┌──────┴──────────────────────────────────────┐    │
│  │  Service (LoadBalancer) → port 5000          │    │
│  └─────────────────────────────────────────────┘    │
└─────────────────────────────────────────────────────┘
              ↑
       Requêtes externes
```

> 🎬 **Netflix** : Netflix utilise Kubernetes pour gérer **10 000+ conteneurs** simultanément (vidéo, recommandations, paiement). Sans K8s, impossible à gérer.

<a id='11'></a>
## 11. 🧪 TP 2.5 — Manifests Kubernetes

In [ ]:
# ============================================================
# TP 2.5 — GÉNÉRER LES MANIFESTS KUBERNETES
# ============================================================

os.makedirs('kubernetes', exist_ok=True)

# ---- deployment.yaml ----
deployment_yaml = '''# deployment.yaml — TP 2.5 — M1MLOP — Amadou MAIGA
# Déployer : kubectl apply -f deployment.yaml
# Vérifier : kubectl get pods

apiVersion: apps/v1
kind: Deployment
metadata:
  name: ml-api
  labels:
    app: ml-api
    version: "1.0"
    module: m1mlop
spec:
  replicas: 3                        # 3 copies du modèle en parallèle
  selector:
    matchLabels:
      app: ml-api
  strategy:
    type: RollingUpdate              # Déploiement sans downtime
    rollingUpdate:
      maxSurge: 1                    # Ajouter max 1 pod à la fois
      maxUnavailable: 0              # Toujours garder 3 pods actifs
  template:
    metadata:
      labels:
        app: ml-api
    spec:
      containers:
      - name: ml-api
        image: ml-api:1.0
        ports:
        - containerPort: 5000
        resources:
          requests:
            memory: "256Mi"          # Minimum garanti
            cpu: "250m"              # 0.25 CPU garanti
          limits:
            memory: "512Mi"          # Maximum autorisé
            cpu: "500m"              # 0.5 CPU maximum
        # Health checks automatiques
        livenessProbe:
          httpGet:
            path: /health
            port: 5000
          initialDelaySeconds: 30
          periodSeconds: 10
        readinessProbe:
          httpGet:
            path: /health
            port: 5000
          initialDelaySeconds: 5
          periodSeconds: 5
        env:
        - name: MODEL_VERSION
          value: "1.0"
'''

# ---- service.yaml ----
service_yaml = '''# service.yaml — Load Balancer pour l\'API ML
apiVersion: v1
kind: Service
metadata:
  name: ml-api-service
spec:
  type: LoadBalancer                 # Expose vers l\'extérieur
  selector:
    app: ml-api
  ports:
  - protocol: TCP
    port: 80                         # Port externe
    targetPort: 5000                 # Port du conteneur
'''

# ---- hpa.yaml ----
hpa_yaml = '''# hpa.yaml — Auto-scaling horizontal
# Si CPU > 70% → lancer plus de pods automatiquement
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: ml-api-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: ml-api
  minReplicas: 3                     # Minimum 3 pods
  maxReplicas: 10                    # Maximum 10 pods
  metrics:
  - type: Resource
    resource:
      name: cpu
      target:
        type: Utilization
        averageUtilization: 70       # Scale si CPU > 70%
  - type: Resource
    resource:
      name: memory
      target:
        type: Utilization
        averageUtilization: 80       # Scale si RAM > 80%
'''

with open('kubernetes/deployment.yaml', 'w') as f:
    f.write(deployment_yaml)
with open('kubernetes/service.yaml', 'w') as f:
    f.write(service_yaml)
with open('kubernetes/hpa.yaml', 'w') as f:
    f.write(hpa_yaml)

print("✅ Manifests Kubernetes générés :")
print("   kubernetes/deployment.yaml  (3 replicas + rolling update)")
print("   kubernetes/service.yaml     (LoadBalancer)")
print("   kubernetes/hpa.yaml         (auto-scaling CPU/RAM)")
print()
print("📋 Commandes kubectl :")
print("   kubectl apply -f kubernetes/")
print("   kubectl get pods")
print("   kubectl get services")
print("   kubectl scale deployment ml-api --replicas=5")
print("   kubectl rollout status deployment ml-api")

<a id='12'></a>
## 12. 📝 Résumé et Bilan du Jour 2

### ✅ Ce que tu as fait aujourd'hui

| Concept | Ce que tu as réalisé |
|---|---|
| **Pipeline séquentiel** | 4 tasks : extract → preprocess → train → validate |
| **Pipeline parallèle** | 3 modèles entraînés en simultané, meilleur sélectionné |
| **Tests CI/CD** | 9 tests (unit + model + integration) avec pytest |
| **API Flask** | Endpoint `/predict`, `/health`, `/info` |
| **Docker** | Dockerfile + conteneurisation du modèle |
| **Kubernetes** | Deployment (3 replicas) + Service + HPA (auto-scaling) |

### 🎯 Points clés

1. **Airflow** orchestre les pipelines ML (scheduling, parallélisation, retries)
2. **Tests automatisés** garantissent la qualité du code et du modèle
3. **Docker** empaquète le modèle avec toutes ses dépendances
4. **Kubernetes** gère la scalabilité et la résilience en production
5. **CI/CD** automatise : test → build → deploy → monitor

### 🚀 Demain — Jour 3
Ton modèle est maintenant **en production**. Comment détecter quand il dégrade ?  
→ **Monitoring, Drift Detection, A/B Testing, Gouvernance RGPD**

---

### 📁 Fichiers générés
```
jour2/
├── jour2_pipelines_orchestration.ipynb
├── airflow_dags/
│   └── dag_tp21_sequential.py
├── docker_app/
│   ├── app.py
│   ├── model.pkl
│   ├── Dockerfile
│   └── requirements.txt
├── kubernetes/
│   ├── deployment.yaml
│   ├── service.yaml
│   └── hpa.yaml
├── test_pipeline.py
└── dag_parallel_visualization.png
```

In [ ]:
# ============================================================
# ✅ VÉRIFICATION FINALE — Bilan du Jour 2
# ============================================================

print("=" * 65)
print("         🎉 BILAN JOUR 2 — PIPELINES & ORCHESTRATION")
print("=" * 65)
print()

checks = [
    ("Pipeline séquentiel (4 tasks)",  os.path.exists('/tmp/model.pkl')),
    ("Pipeline parallèle (3 modèles)", os.path.exists('/tmp/best_model.pkl')),
    ("DAG Airflow généré",             os.path.exists('airflow_dags/dag_tp21_sequential.py')),
    ("Tests pytest générés",           os.path.exists('test_pipeline.py')),
    ("API Flask générée",              os.path.exists('docker_app/app.py')),
    ("Dockerfile généré",              os.path.exists('docker_app/Dockerfile')),
    ("K8s deployment.yaml",            os.path.exists('kubernetes/deployment.yaml')),
    ("K8s service.yaml",               os.path.exists('kubernetes/service.yaml')),
    ("K8s hpa.yaml",                   os.path.exists('kubernetes/hpa.yaml')),
]

all_ok = True
for label, condition in checks:
    status = "✅" if condition else "❌"
    if not condition:
        all_ok = False
    print(f"  {status} {label}")

print()
print("=" * 65)
if all_ok:
    print("  🏆 Tous les TPs du Jour 2 sont COMPLÉTÉS !")
else:
    print("  ⚠️  Certains éléments manquent, relance les cellules.")
print("=" * 65)
print()
print("  📌 Git commit suggéré :")
print("  git add jour2/")
print('  git commit -m "✅ Jour 2 - Airflow Pipelines + Docker + K8s"')
print("  git push")
print("=" * 65)